# 00 - Create SWSAcademy Demo Jobs

Run this notebook **once** after importing the SWSAcademy course folder. It creates one Lakeflow job per
Machine Learning folder that runs all of that folder's **demos** in order (labs are not included):

| Job | What it runs |
|---|---|
| SWSAcademy - 01 Data Preparation | Admin setup, load and explore, imputation, Spark ML feature pipeline, Feature Store |
| SWSAcademy - 02 Model Development | Admin setup, supervised, unsupervised, MLflow tracking, Optuna tuning, Genie Code setup |
| SWSAcademy - 03 Model Deployment | Admin setup, batch, pipeline (Lakeflow) and real-time deployment |
| SWSAcademy - 04 ML Operations | Admin setup, multi-task jobs (UI, REST API, CLI), parallel model tests, Lakehouse Monitoring |
| SWSAcademy - 99 Cleanup and Reset | Removes everything the demos created (dry run unless `confirm_delete = YES`) |

The notebook finds the course notebooks **relative to its own location**, so it works wherever the folder was imported.
Running it again updates the jobs instead of creating duplicates.

**After it finishes:** open **Jobs & Pipelines** and run the jobs **one at a time, in order 01 → 02 → 03 → 04**.
Job 01 creates the `swsacademy` catalog that the other jobs use. All jobs run on serverless compute and have no schedule.

In [ ]:
dbutils.widgets.text("name_suffix", "", "Job name suffix (optional, e.g. your initials)")
dbutils.widgets.dropdown("pin_environment", "YES", ["YES", "NO"], "Set serverless environment v5 on course notebooks")

In [ ]:
import base64, json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
ROOT = here.rsplit("/", 1)[0]
ML = f"{ROOT}/01 Machine Learning"
SUFFIX = dbutils.widgets.get("name_suffix").strip()
TAGS = {"course": "swsacademy", "purpose": "instructor-demo"}

F1 = f"{ML}/1 - swsacademy-data-preparation-for-machine-learning/Data Preparation for Machine Learning"
F2 = f"{ML}/2 - swsacademy-machine-learning-model-development"
F3 = f"{ML}/3 - swsacademy-machine-learning-model-deployment"
F4 = f"{ML}/4 - swsacademy-machine-learning-operations"

def exists(path):
    try:
        w.workspace.get_status(path)
        return True
    except Exception:
        return False

missing = [p for p in (F1, F2, F3, F4) if not exists(p)]
if missing:
    raise FileNotFoundError("These course folders were not found next to this notebook:\n  " + "\n  ".join(missing)
                            + "\nKeep this notebook in the top folder that contains '01 Machine Learning'.")
print("Course folder:", ROOT)

## 1. Set the serverless environment version

The demos use Spark ML and Python 3.12 features that need **serverless environment version 5**. A folder import does not
always keep each notebook's environment setting, so this step writes it into every course notebook. It takes a few minutes.

In [ ]:
def notebooks_under(path):
    for obj in w.workspace.list(path):
        if obj.object_type.value == "DIRECTORY":
            yield from notebooks_under(obj.path)
        elif obj.object_type.value == "NOTEBOOK" and obj.language and obj.language.value == "PYTHON":
            yield obj.path

if dbutils.widgets.get("pin_environment") == "YES":
    pinned = 0
    for nb in notebooks_under(ML):
        content = w.api_client.do("GET", "/api/2.0/workspace/export", query={"path": nb, "format": "JUPYTER"})["content"]
        doc = json.loads(base64.b64decode(content))
        meta = doc["metadata"].setdefault("application/vnd.databricks.v1+notebook", {})
        if (meta.get("environmentMetadata") or {}).get("environment_version") == "5":
            continue
        meta["environmentMetadata"] = {"base_environment": "", "environment_version": "5"}
        w.api_client.do("POST", "/api/2.0/workspace/import", body={
            "path": nb, "format": "JUPYTER", "overwrite": True,
            "content": base64.b64encode(json.dumps(doc).encode()).decode()})
        pinned += 1
    print(f"Environment version 5 set on {pinned} notebooks (others already had it).")
else:
    print("Skipped.")

## 2. Create or update the jobs

In [ ]:
def nb(key, path, deps=()):
    task = {"task_key": key, "notebook_task": {"notebook_path": path, "source": "WORKSPACE"}, "timeout_seconds": 3600}
    if deps:
        task["depends_on"] = [{"task_key": d} for d in deps]
    return task

def chain(items):
    tasks, prev = [], None
    for key, path in items:
        tasks.append(nb(key, path, [prev] if prev else ()))
        prev = key
    return tasks

p11 = f"{F4}/M01 - Modern MLOps/1.1 Demo Pipeline - Data Quality and Feature Analysis"
p22 = f"{F4}/M02 - Architecting MLOps Solutions/2.2 Demo Pipeline - Testing Model Performance"
rt = f"{F3}/M04 - Real-time Deployment and Online Stores"

JOBS = [
    ("SWSAcademy - 01 Data Preparation",
     "Folder 1 demos: load and explore, imputation, Spark ML feature pipeline, Feature Store.",
     chain([
        ("00_admin_setup", f"{F1}/00 - SWSAcademy Admin Setup"),
        ("1_1_load_and_explore", f"{F1}/M01 - Managing and Exploring Data/1.1 Demo - Load and Explore Data"),
        ("2_1_imputation_transformation", f"{F1}/M02 - Data Preparation and Feature Engineering/2.1 Demo - Data Imputation and Transformation Pipeline"),
        ("2_2_feature_engineering_pipeline", f"{F1}/M02 - Data Preparation and Feature Engineering/2.2 Demo - Build a Feature Engineering Pipeline"),
        ("3_1_feature_store", f"{F1}/M03 - Feature Store/3.1 Demo - Using Feature Store for Feature Engineering"),
     ])),
    ("SWSAcademy - 02 Model Development",
     "Folder 2 demos: supervised, unsupervised, MLflow tracking, Optuna tuning, Genie Code setup.",
     chain([
        ("00_admin_setup", f"{F2}/00 - SWSAcademy Admin Setup"),
        ("1_1a_supervised_learning", f"{F2}/M01 - Model Development Workflow/1.1a Demo - Supervised Learning"),
        ("1_1b_unsupervised_learning", f"{F2}/M01 - Model Development Workflow/1.1b Demo - Unsupervised Learning"),
        ("1_2_mlflow_tracking", f"{F2}/M01 - Model Development Workflow/1.2 Demo - Model Tracking with MLflow"),
        ("2_1_optuna_tuning", f"{F2}/M02 - Hyperparameter Tuning/2.1 Demo - Hyperparameter Tuning with Optuna"),
        ("3_1_genie_code_setup", f"{F2}/M03 - Agentic Machine Learning/3.1 Demo - Model Development with Genie Code"),
     ])),
    ("SWSAcademy - 03 Model Deployment",
     "Folder 3 demos: batch, pipeline (Lakeflow) and real-time deployment with Model Serving.",
     chain([
        ("00_admin_setup", f"{F3}/00 - SWSAcademy Admin Setup"),
        ("2_1_batch_deployment", f"{F3}/M02 - Batch Deployment/2.1 Demo - Batch Deployment"),
        ("3_1a_register_pipeline_model", f"{F3}/M03 - Pipeline Deployment/3.1a Demo - Pipeline Deployment"),
        ("3_1b_inference_pipeline", f"{F3}/M03 - Pipeline Deployment/3.1b Run Inference Pipeline (job step)"),
        ("4_1_realtime_ab_endpoint", f"{rt}/4.1 Demo - Real-time Deployment with Model Serving"),
        ("4_2_custom_model_endpoint", f"{rt}/4.2 Demo - Custom Model Deployment with Model Serving"),
     ])),
    ("SWSAcademy - 04 ML Operations",
     "Folder 4 demos: multi-task jobs (UI, REST API, CLI), parallel model tests, Lakehouse Monitoring.",
     chain([
        ("00_admin_setup", f"{F4}/00 - SWSAcademy Admin Setup"),
        ("1_1a_data_quality", f"{p11}/1.1a - Data Quality Assessment"),
        ("1_1b_unusual_patterns_alert", f"{p11}/1.1b - Alert on Unusual Patterns"),
        ("1_1c_investigate_patterns", f"{p11}/1.1c -Investigate and Resolve Unusual Patterns"),
        ("1_1d_feature_importance", f"{p11}/1.1d - Feature Importance Analysis"),
        ("1_1e_save_report", f"{p11}/1.1e - Save Report Notebook (Success Path)"),
        ("2_1_job_via_rest_api", f"{F4}/M02 - Architecting MLOps Solutions/2.1 Demo - Machine Learning Pipeline with Databricks API"),
        ("2_2_cli_job_config", f"{F4}/M02 - Architecting MLOps Solutions/2.2 Demo - Model Testing Job with the Databricks CLI"),
        ("2_2a_train_and_deploy", f"{p22}/2.2a Train and Deploy Model"),
     ]) + [
        nb("2_2b_performance_test", f"{p22}/2.2b Parallel Task 1 - Performance Test", ["2_2a_train_and_deploy"]),
        nb("2_2b_prediction_analysis", f"{p22}/2.2b Parallel Task 2 - Model Prediction Analysis", ["2_2a_train_and_deploy"]),
        nb("2_2c_summary_report", f"{p22}/2.2c Model Summary Report", ["2_2b_performance_test", "2_2b_prediction_analysis"]),
        nb("3_1_lakehouse_monitoring", f"{F4}/M03 - Implementation and Monitoring MLOps Solution/3.1 Demo - Lakehouse Monitoring Dashboard", ["2_2c_summary_report"]),
     ]),
]

cleanup_nb = next((p for p in (f"{ROOT}/99 - SWSAcademy Cleanup and Billing",) if exists(p)), None)

# check every notebook a job points at before creating anything
bad = [t["notebook_task"]["notebook_path"] for _, _, tasks in JOBS for t in tasks if not exists(t["notebook_task"]["notebook_path"])]
if bad:
    raise FileNotFoundError("Notebooks not found:\n  " + "\n  ".join(bad))

existing = {j.settings.name: j.job_id for j in w.jobs.list() if j.settings and j.settings.name}
host = w.config.host.rstrip("/")
results = []

def upsert(name, settings):
    if name in existing:
        w.api_client.do("POST", "/api/2.1/jobs/reset", body={"job_id": existing[name], "new_settings": settings})
        return existing[name], "updated"
    return w.api_client.do("POST", "/api/2.1/jobs/create", body=settings)["job_id"], "created"

for name, desc, tasks in JOBS:
    full = f"{name}{' ' + SUFFIX if SUFFIX else ''}"
    settings = {"name": full, "description": desc, "tags": TAGS, "max_concurrent_runs": 1,
                "queue": {"enabled": True}, "tasks": tasks}
    results.append((full, len(tasks)) + upsert(full, settings))

if cleanup_nb:
    full = f"SWSAcademy - 99 Cleanup and Reset{' ' + SUFFIX if SUFFIX else ''}"
    settings = {"name": full, "tags": {"course": "swsacademy", "purpose": "cleanup"}, "max_concurrent_runs": 1,
                "description": "Removes everything the SWSAcademy demos created. Dry run by default; run with confirm_delete = YES to delete.",
                "parameters": [{"name": "confirm_delete", "default": "NO"}, {"name": "billing_days", "default": "7"}],
                "tasks": [nb("cleanup_and_billing", cleanup_nb)]}
    results.append((full, 1) + upsert(full, settings))

html = "<table><tr><th style='text-align:left'>Job</th><th>Tasks</th><th>Status</th></tr>" + "".join(
    f"<tr><td><a href='{host}/jobs/{jid}' target='_blank'>{n}</a></td><td style='text-align:center'>{k}</td><td>{s}</td></tr>"
    for n, k, jid, s in results) + "</table>"
displayHTML(html + "<p>Run the jobs one at a time: <b>01 → 02 → 03 → 04</b>. Job 01 must run first.</p>")